In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)


In [3]:
base_dir = Path(r"C:\Users\matth\Downloads\Maven+Fuzzy+Factory")

file_map = {
    "website_sessions": base_dir / "website_sessions.csv",
    "website_pageviews": base_dir / "website_pageviews.csv",
    "orders": base_dir / "orders.csv",
    "order_items": base_dir / "order_items.csv",
    "order_item_refunds": base_dir / "order_item_refunds.csv",
    "products": base_dir / "products.csv",
    "data_dictionary": base_dir / "maven_fuzzy_factory_data_dictionary.csv",
}


for name, path in file_map.items():
    print(f"{name:22} -> {path} | exists={path.exists()}")


website_sessions       -> C:\Users\matth\Downloads\Maven+Fuzzy+Factory\website_sessions.csv | exists=True
website_pageviews      -> C:\Users\matth\Downloads\Maven+Fuzzy+Factory\website_pageviews.csv | exists=True
orders                 -> C:\Users\matth\Downloads\Maven+Fuzzy+Factory\orders.csv | exists=True
order_items            -> C:\Users\matth\Downloads\Maven+Fuzzy+Factory\order_items.csv | exists=True
order_item_refunds     -> C:\Users\matth\Downloads\Maven+Fuzzy+Factory\order_item_refunds.csv | exists=True
products               -> C:\Users\matth\Downloads\Maven+Fuzzy+Factory\products.csv | exists=True
data_dictionary        -> C:\Users\matth\Downloads\Maven+Fuzzy+Factory\maven_fuzzy_factory_data_dictionary.csv | exists=True


In [4]:
website_sessions = pd.read_csv(file_map["website_sessions"])
website_pageviews = pd.read_csv(file_map["website_pageviews"])
orders = pd.read_csv(file_map["orders"])
order_items = pd.read_csv(file_map["order_items"])
order_item_refunds = pd.read_csv(file_map["order_item_refunds"])
products = pd.read_csv(file_map["products"])

tables = {
    "website_sessions": website_sessions,
    "website_pageviews": website_pageviews,
    "orders": orders,
    "order_items": order_items,
    "order_item_refunds": order_item_refunds,
    "products": products,
}


In [5]:
for name, df in tables.items():
    print("=" * 80)
    print(name)
    print(df.shape)
    print(df.head(3))
    print(df.dtypes)


website_sessions
(472871, 9)
   website_session_id           created_at  user_id  is_repeat_session  \
0                   1  2012-03-19 08:04:16        1                  0   
1                   2  2012-03-19 08:16:49        2                  0   
2                   3  2012-03-19 08:26:55        3                  0   

  utm_source utm_campaign utm_content device_type             http_referer  
0    gsearch     nonbrand      g_ad_1      mobile  https://www.gsearch.com  
1    gsearch     nonbrand      g_ad_1     desktop  https://www.gsearch.com  
2    gsearch     nonbrand      g_ad_1     desktop  https://www.gsearch.com  
website_session_id     int64
created_at            object
user_id                int64
is_repeat_session      int64
utm_source            object
utm_campaign          object
utm_content           object
device_type           object
http_referer          object
dtype: object
website_pageviews
(1188124, 4)
   website_pageview_id           created_at  website_session

In [6]:
for name, df in tables.items():
    print("=" * 80)
    print(name)
    print("Missing values:")
    print(df.isna().sum())


website_sessions
Missing values:
website_session_id        0
created_at                0
user_id                   0
is_repeat_session         0
utm_source            83328
utm_campaign          83328
utm_content           83328
device_type               0
http_referer          39917
dtype: int64
website_pageviews
Missing values:
website_pageview_id    0
created_at             0
website_session_id     0
pageview_url           0
dtype: int64
orders
Missing values:
order_id              0
created_at            0
website_session_id    0
user_id               0
primary_product_id    0
items_purchased       0
price_usd             0
cogs_usd              0
dtype: int64
order_items
Missing values:
order_item_id      0
created_at         0
order_id           0
product_id         0
is_primary_item    0
price_usd          0
cogs_usd           0
dtype: int64
order_item_refunds
Missing values:
order_item_refund_id    0
created_at              0
order_item_id           0
order_id                0


In [7]:
# Duplicate key checks
print("Duplicate website_session_id:", website_sessions["website_session_id"].duplicated().sum())
print("Duplicate website_pageview_id:", website_pageviews["website_pageview_id"].duplicated().sum())
print("Duplicate order_id:", orders["order_id"].duplicated().sum())
print("Duplicate order_item_id:", order_items["order_item_id"].duplicated().sum())
print("Duplicate order_item_refund_id:", order_item_refunds["order_item_refund_id"].duplicated().sum())


Duplicate website_session_id: 0
Duplicate website_pageview_id: 0
Duplicate order_id: 0
Duplicate order_item_id: 0
Duplicate order_item_refund_id: 0


In [8]:
# Relationship checks
orphan_orders = orders.loc[~orders["website_session_id"].isin(website_sessions["website_session_id"])]
orphan_pageviews = website_pageviews.loc[~website_pageviews["website_session_id"].isin(website_sessions["website_session_id"])]
orphan_order_items = order_items.loc[~order_items["order_id"].isin(orders["order_id"])]
orphan_refunds = order_item_refunds.loc[~order_item_refunds["order_item_id"].isin(order_items["order_item_id"])]

print("Orphan orders:", len(orphan_orders))
print("Orphan pageviews:", len(orphan_pageviews))
print("Orphan order_items:", len(orphan_order_items))
print("Orphan refunds:", len(orphan_refunds))


Orphan orders: 0
Orphan pageviews: 0
Orphan order_items: 0
Orphan refunds: 0


In [9]:
datetime_cols = {
    "website_sessions": ["created_at"],
    "website_pageviews": ["created_at"],
    "orders": ["created_at"],
    "order_items": ["created_at"],
    "order_item_refunds": ["created_at"],
    "products": ["created_at"],
}

for table_name, cols in datetime_cols.items():
    df = tables[table_name]
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

numeric_cols = {
    "orders": ["price_usd", "cogs_usd"],
    "order_items": ["price_usd", "cogs_usd"],
    "order_item_refunds": ["refund_amount_usd"],
}

for table_name, cols in numeric_cols.items():
    df = tables[table_name]
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")


In [10]:
page_counts = (
    website_pageviews["pageview_url"]
    .value_counts()
    .rename_axis("pageview_url")
    .reset_index(name="pageviews")
)

page_counts.head(50)


,pageview_url,pageviews
0,/products,261231
1,/the-original-mr-fuzzy,162525
2,/home,137576
3,/lander-2,131170
4,/cart,94953
5,/lander-3,79000
6,/lander-5,68166
7,/shipping,64484
8,/billing-2,48441
9,/lander-1,47574


In [11]:
funnel_related = page_counts[
    page_counts["pageview_url"].str.contains(
        "products|cart|shipping|billing|thank-you",
        case=False,
        na=False,
    )
]

funnel_related


,pageview_url,pageviews
0,/products,261231
4,/cart,94953
7,/shipping,64484
8,/billing-2,48441
10,/thank-you-for-your-order,32313
14,/billing,3617


In [12]:
website_sessions_clean = website_sessions.copy()
website_pageviews_clean = website_pageviews.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()
order_item_refunds_clean = order_item_refunds.copy()
products_clean = products.copy()


In [ ]:
DB_USER = "postgres"
DB_PASSWORD = "password"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "funnel_analysis_project"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)


In [14]:
load_order = [
    ("website_sessions", website_sessions_clean),
    ("website_pageviews", website_pageviews_clean),
    ("orders", orders_clean),
    ("order_items", order_items_clean),
    ("order_item_refunds", order_item_refunds_clean),
    ("products", products_clean),
]

for table_name, df in load_order:
    df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")


Loaded website_sessions: 472,871 rows
Loaded website_pageviews: 1,188,124 rows
Loaded orders: 32,313 rows
Loaded order_items: 40,025 rows
Loaded order_item_refunds: 1,731 rows
Loaded products: 4 rows
